# Sales by Customer — Aggregation Analysis

Processes `input.csv` into a customer-level summary. Generated by pandas-copilot;
assembled verbatim from the working artifacts (`pipeline.py` + `checks.py`) that
passed Gate B. Rerunnable on any new file with the same structure — every check
re-fires on rerun.

## Task
- Group orders by customer
- Sum amount per customer, count orders per customer

## Validation plan (agreed at Gate A)
1. Conservation: total amount before and after aggregation must match
2. Conservation: order counts must sum to the number of input rows
3. Output has exactly the agreed columns: customer, total_amount, order_count
4. customer is a unique key — one row per customer, none lost
5. No null values anywhere in the output
6. Output rows are sorted by customer name alphabetically
7. (manual) Spot-check that the top customers' totals look plausible

## Read

In [ ]:
import pandas as pd


def load(path):
    """Read the raw sales orders (CSV here; Excel would carry dtype/na_values guards)."""
    return pd.read_csv(path)


raw = load('input.csv')
print('Input shape:', raw.shape)
raw.head()

## Transform

In [ ]:
def transform(raw):
    """Aggregate orders to one row per customer: total amount + order count."""
    return (
        raw.groupby('customer', as_index=False)
           .agg(total_amount=('amount', 'sum'), order_count=('order_id', 'count'))
           .sort_values('customer')
           .reset_index(drop=True)
    )


result = transform(raw)
result

## Validation: total amount is conserved

Total amount before and after aggregation must match.

In [ ]:
def check_total_amount_conserved(raw, out):
    """Conservation: total amount before and after aggregation must match"""
    raw_sum, out_sum = raw['amount'].sum(), out['total_amount'].sum()
    assert abs(raw_sum - out_sum) < 0.01, f'amount mismatch: raw={raw_sum}, out={out_sum}'


check_total_amount_conserved(raw, result)
print(f"PASS — total amount conserved: {raw['amount'].sum():.2f}")

## Validation: order count is conserved

Order counts must sum to the number of input rows.

In [ ]:
def check_order_count_conserved(raw, out):
    """Conservation: order counts must sum to the number of input rows"""
    assert out['order_count'].sum() == len(raw), (
        f"order count mismatch: sum={out['order_count'].sum()}, input rows={len(raw)}"
    )


check_order_count_conserved(raw, result)
print(f'PASS — order counts sum to {len(raw)} input rows')

## Validation: column structure

Output has exactly the agreed columns: customer, total_amount, order_count.

In [ ]:
def check_column_structure(raw, out):
    """Output has exactly the agreed columns: customer, total_amount, order_count"""
    expected = {'customer', 'total_amount', 'order_count'}
    assert set(out.columns) == expected, f'columns: expected {expected}, got {set(out.columns)}'


check_column_structure(raw, result)
print('PASS — columns:', list(result.columns))

## Validation: customer is a unique key

One row per customer, and no customer lost during aggregation.

In [ ]:
def check_customer_unique(raw, out):
    """customer is a unique key — one row per customer, none lost"""
    assert out['customer'].is_unique, 'duplicate customers in output'
    assert set(out['customer']) == set(raw['customer']), 'customer set changed during aggregation'


check_customer_unique(raw, result)
print(f"PASS — {result['customer'].nunique()} unique customers")

## Validation: no nulls

No null values anywhere in the output.

In [ ]:
def check_no_nulls(raw, out):
    """No null values anywhere in the output"""
    nulls = out.isna().sum()
    assert nulls.sum() == 0, f'nulls detected: {nulls[nulls > 0].to_dict()}'


check_no_nulls(raw, result)
print('PASS — no nulls')

## Validation: sorted by customer

Output rows are sorted by customer name alphabetically.

In [ ]:
def check_sorted_by_customer(raw, out):
    """Output rows are sorted by customer name alphabetically"""
    assert out['customer'].is_monotonic_increasing, 'output not sorted by customer'


check_sorted_by_customer(raw, result)
print('PASS — sorted by customer')

## Requires manual confirmation

- [ ] Spot-check that the top customers' totals look plausible for the business

Statistics for eyeballing:

In [ ]:
result.describe()

## Save

In [ ]:
# Save the result to its own output file. expected-output.csv is the example's
# reference oracle — compare against it, never overwrite it. (Real deliverables
# have no oracle file; this cell would just save.)
result.to_csv('sales-by-customer.csv', index=False)

expected = pd.read_csv('expected-output.csv')
pd.testing.assert_frame_equal(
    result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False
)
print('Result saved to sales-by-customer.csv and matches expected-output.csv')
result